# HEILO Seed v2.0: motor novo (técnicas do estilo Gemma), treinado do zero

**O que muda em relação à v1.x:** só o "motor" do modelo. Continua 100% da HEILO, treinado do zero, sem nenhum peso ou dado do Google. Só as **técnicas** vêm do Gemma:

- **RoPE:** posição "girando" junto com o texto, que entende melhor a ordem das palavras.
- **RMSNorm:** normalização mais simples e estável.
- **GeGLU:** filtro com "porta" em cada camada, que aprende mais com o mesmo tamanho.
- **GQA:** as cabeças de atenção compartilham memória, então responde mais rápido no PC.
- **QK-norm:** treino mais estável.

**Teste antes de subir para cá:** em um treino pequeno igual para as duas (mesmos dados, mesmo tamanho, mesmos passos), a arquitetura nova terminou com perda de validação **3,16 contra 3,65** da antiga, e ainda com 7% menos parâmetros.

**Receita:** o mesmo tokenizer e o mesmo corpus grande da v1.1, que já estão no Drive. O pré-treino é do zero, com mais tempo (210 min). Depois vem o ajuste de conversa misturado, com os exemplos aprovados (inclusive os novos que você revisou), e a avaliação na mesma régua.

**Antes:** rode `publicar_heilo.bat` no PC, para os exemplos aprovados irem para o GitHub. Aqui no Colab: GPU T4 e o secret `GITHUB_TOKEN`. Depois, `Executar tudo`. Leva cerca de 4h30. Se cair, rode tudo de novo: ele continua de onde parou.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v2.0"
MINUTOS_PRETREINO = 210
MAX_CARACTERES = 1_200_000_000      # só usado se o corpus grande ainda não existir no Drive
LOTE = 32
PASSOS_SFT = 3000
import torch, time, os, sys, json, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive: tokenizer e corpus da v1.x; checkpoints da v2 numa pasta própria
from google.colab import drive
drive.mount("/content/drive")
PASTA_V1 = "/content/drive/MyDrive/heilo_seed_v1"
PASTA = "/content/drive/MyDrive/heilo_seed_v2"
os.makedirs(PASTA, exist_ok=True)
assert os.path.exists(f"{PASTA_V1}/tokenizer.json"), "Não achei o tokenizer da v1 no Drive (pasta heilo_seed_v1)."

In [ ]:
# 3) Código da HEILO (versão mais nova do GitHub, com o motor v2 e os dados aprovados)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
!git -C /content/heilo log --oneline -1
from heilo.models.seed.gpt import GPTConfig
assert "arquitetura" in GPTConfig.__dataclass_fields__, "O GitHub ainda não tem o motor v2: rode publicar_heilo.bat no PC."

In [ ]:
# 4) Tokenizer e corpus (reaproveita os da v1.x)
from pathlib import Path
from heilo.training import seed_v1, pretrain
from heilo.models.seed.tokenizer import BPETokenizer
tok = BPETokenizer.from_json(Path(PASTA_V1, "tokenizer.json").read_text(encoding="utf-8"))
corpus_dir = Path(PASTA_V1) / "corpus_grande"
if not (corpus_dir / "meta.json").exists():
    corpus_dir = Path(PASTA) / "corpus"
    if not (corpus_dir / "meta.json").exists():
        t0 = time.time()
        pretrain.preparar_corpus(seed_v1.textos_wikipedia(MAX_CARACTERES), tok, corpus_dir,
                                 max_tokens=10**12, log=lambda s: None)
        print(f"corpus pronto em {(time.time()-t0)/60:.1f} min")
print(open(corpus_dir / "meta.json").read())

## Pré-treino do zero com o motor novo

In [ ]:
# 5) Mede a velocidade e ajusta os passos ao tempo
cfg = seed_v1.CONFIG_V2
ck = Path(PASTA) / "pretreino.pt"
if not Path(PASTA, "passos.txt").exists():
    Path("/content/vel_v2.pt").unlink(missing_ok=True)
    t0 = time.time()
    pretrain.pretreinar(cfg, tok, corpus_dir, Path("/content/vel_v2.pt"), passos=60, lote=LOTE,
                        avaliar_cada=10**9, salvar_cada=10**9, aquecimento=10, log=lambda s: None)
    seg = (time.time() - t0) / 60
    Path(PASTA, "passos.txt").write_text(str(int(MINUTOS_PRETREINO * 60 / seg)))
    print(f"{seg:.3f} s/passo")
PASSOS = int(Path(PASTA, "passos.txt").read_text())
print(PASSOS, "passos")

In [ ]:
# 6) Pré-treino (retoma sozinho se o Colab cair)
res_pre = pretrain.pretreinar(cfg, tok, corpus_dir, ck, passos=PASSOS, lote=LOTE, lr=8e-4, lr_min=8e-5,
                              tempo_max_min=MINUTOS_PRETREINO + 20, avaliar_cada=500, salvar_cada=1000)
print("pré-treino:", res_pre["passo"], "passos |", res_pre["hist"][-1])

## Ajuste de conversa + avaliação

In [ ]:
# 7) SFT misturado com texto (não esquece o português, não gruda em frases prontas)
from heilo.training.pipeline import TrainingPipeline
pipe = TrainingPipeline()
split = seed_v1.dividir_sft(pipe)
print(split["manifest"])
modelo = res_pre["modelo"]
res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=PASSOS_SFT, lote=LOTE,
                                    lr=5e-5, paciencia=5, pasta_corpus=corpus_dir, peso_texto=0.5)
arq = pretrain.exportar(modelo, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                        {"versao": NOME_VERSAO, "arquitetura": "moderna (RoPE, RMSNorm, GeGLU, GQA, QK-norm)",
                         "passos_pretreino": res_pre["passo"], "hist_pretreino": res_pre["hist"], "sft": res_sft,
                         "corpus": json.loads((corpus_dir / "meta.json").read_text())})
print("exportado:", arq, round(arq.stat().st_size / 1e6, 1), "MB")

In [ ]:
# 8) Avaliação na mesma régua + decisão automática
from heilo.training.ciclos import CiclosSeed
ciclos = CiclosSeed(pipe)
dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                        {"arquitetura": "moderna", "pretreino": {"passos": res_pre["passo"], "minutos": res_pre["minutos"],
                                       "ultimo": res_pre["hist"][-1] if res_pre["hist"] else None},
                         "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val")}},
                        origem="motor novo (estilo Gemma: RoPE, RMSNorm, GeGLU, GQA, QK-norm), do zero + SFT misturado")
print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
from heilo.models.seed.gpt import MiniGPTChat
chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
for q in ["oi, tudo bem?", "quem é você?", "qual é a capital de portugal?", "o que é uma função em python?",
          "o que faz o g04?", "me fala sobre o Brasil", "Se hoje é segunda, amanhã é que dia?"]:
    print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7,
                                                  penalidade=1.2, sem_repetir=4), "\n")

In [ ]:
# 9) Salvar no GitHub
!cd /content/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd /content/heilo && git add -f models/seed/versions/{NOME_VERSAO} data/training/versions.json data/training/eval && git add models/seed/weights && git commit -q -m "HEILO Seed {NOME_VERSAO}: motor novo estilo Gemma, do zero (Colab)" || echo "nada novo"
!cd /content/heilo && git stash -u -q; git pull --rebase && git push && echo "ENVIADO PARA O GITHUB"